# Final Public Export (LLM-labeled goldset)

This notebook builds two public JSON artifacts:
- Labeled goldset (queries + labeled docs).
- EPMC documents JSONL (title/abstract store).

Inputs:
- output/dicty_gold_build/5a_gold_query_expand.parquet (columns: query, query_expand_synonyms, query_expand_long, docs; or query, query_expand, docs for backward compat)
- output/dicty_gold_build/6d_llm_full_agreement.tsv
- output/dicty_gold_build/3_articles_cleaned_abstract.parquet

Output JSONL fields per question: query_id, query_text, body_expansion_synonyms, body_expansion_long, documents, docs.

Outputs:
- output/dicty_gold_build/7b_dicty_gold_llm_private.jsonl (full payload, all fields)
- output/dicty_gold_build/7a_dicty_gold_llm_public.jsonl (canonical pipeline keys)
- output/dicty_gold_build/7c_articles_cleaned_abstract.jsonl

In [1]:
from pathlib import Path
import json
import polars as pl

## Load inputs

In [2]:
GOLD_PATH = Path("../output/dicty_gold_build/5a_gold_query_expand.parquet")
LABELS_PATH = Path("../output/dicty_gold_build/6d_llm_full_agreement.tsv")
DOCS_PATH = Path("../output/dicty_gold_build/3_articles_cleaned_abstract.parquet")
OUT_JSONL = Path("../output/dicty_gold_build/7a_dicty_gold_llm_public.jsonl")
OUT_JSONL_PRIVATE = Path("../output/dicty_gold_build/7b_dicty_gold_llm_private.jsonl")
DOCS_JSONL_OUT = Path("../output/dicty_gold_build/7c_articles_cleaned_abstract.jsonl")

def load_labels(path: Path) -> pl.DataFrame:
    if path.suffix == ".jsonl":
        df = pl.read_ndjson(path)
    elif path.suffix == ".tsv":
        df = pl.read_csv(path, separator="\t")
    else:
        raise ValueError(f"Unsupported labels format: {path.suffix}")

    if "reason" not in df.columns:
        df = df.with_columns(pl.lit("").alias("reason"))

    return df.with_columns([
        pl.col("group_claim_id").cast(pl.Utf8),
        pl.col("pmid").cast(pl.Utf8),
    ])

gold = pl.read_parquet(GOLD_PATH)
labels = load_labels(LABELS_PATH).unique(subset=["group_claim_id", "pmid"])

gold.head(2)

group_claim_id,rep_claim_id,query,n_variants,n_citations,query_n_words,years,docs,query_expand_synonyms,query_expand_long,query_expand
i64,i64,str,u32,u32,u32,list[i32],list[struct[7]],str,str,str
1,1,"""A basic region in the tail is …",1,1,17,[2014],"[{13954,""24747353"",""The association of myosin IB with actin waves in dictyostelium requires both the plasma membrane-binding site and actin-binding region in the myosin tail."",""F-actin structures and their distribution are important determinants of the dynamic shapes and functions of eukaryotic cells. Actin waves are F-actin formations that move along the ventral cell membrane driven by actin polymerization. Dictyostelium myosin IB is associated with actin waves but its role in the wave is unknown. Myosin IB is a monomeric, non-filamentous myosin with a globular head that binds to F-actin and has motor activity, and a non-helical tail comprising a basic region, a glycine-proline-glutamine-rich region and an SH3-domain. The basic region binds to acidic phospholipids in the plasma membrane through a short basic-hydrophobic site and the Gly-Pro-Gln region binds F-actin. In the current work we found that both the basic-hydrophobic site in the basic region and the Gly-Pro-Gln region of the tail are required for the association of myosin IB with actin waves. This is the first evidence that the Gly-Pro-Gln region is required for localization of myosin IB to a specific actin structure in situ. The head is not required for myosin IB association with actin waves but binding of the head to F-actin strengthens the association of myosin IB with waves and stabilizes waves. Neither the SH3-domain nor motor activity is required for association of myosin IB with actin waves. We conclude that myosin IB contributes to anchoring actin waves to the plasma membranes by binding of the basic-hydrophobic site to acidic phospholipids in the plasma membrane and binding of the Gly-Pro-Gln region to F-actin in the wave."",2014,[94],[""Brzeska et al. 2014""]}]","""A basic region in the tail is …","""A basic region in the tail is …","""A basic region in the tail is …"
2,2,"""A cDNA clone derived from psvA…",1,1,19,[1983],"[{8316,""6301681"",""Regulation of dictyostelium discoideum mRNAs specific for prespore or prestalk cells."",""Prespore and prestalk cells in Dictyostelium discoideum aggregates can be separated by density gradient centrifugation. Using poly(A+) RNA from the fractionated cells to probe a cDNA library of mRNAs from postaggregation cells, we were able to identify six cDNA clones representing RNAs enriched in prespore or prestalk cells. Remarkably, transcripts of six of seven cDNA clones, previously selected to encode mRNAs present in postaggregating cells but low or absent in growing cells, also are enriched in RNA from either prestalk or prespore cells. By hybridization of cDNA probes to nitrocellulose blots of formaldehyde RNA gels, these 13 mRNA species have been examined with respect to cell type specificity, temporal pattern of accumulation, and affect of disaggregation and cAMP on accumulation. Aggregation-stage mRNAs tend to fit into three different classes. All prespore mRNAs are similar in all aspects of their regulation, while prestalk mRNAs fall into two co-regulated classes. All mRNAs that are present at significant levels during growth and differentiation are found in both cell types at comparable levels. Our results indicate that there is coordinate control of expression of genes specific for the two principal cell types."",1983,[102],[""Barklis and Lodish 1983""]}]","""A cDNA clone derived from psvA…","""A cDNA clone derived from psvA…","""A cDNA clone derived from psvA…"


## Build final public JSON

We join LLM labels to the goldset and keep one JSON output for public release.

In [3]:
if "docs" not in gold.columns:
    raise ValueError("Expected 'docs' column in gold_with_query_expand.parquet")

# Prefer query_expand_synonyms / query_expand_long; fallback to query_expand for backward compat
select_cols = ["group_claim_id", "query", "docs"]
if "query_expand_synonyms" in gold.columns and "query_expand_long" in gold.columns:
    select_cols.extend(["query_expand_synonyms", "query_expand_long"])
else:
    gold = gold.with_columns([
        pl.col("query_expand").alias("query_expand_synonyms"),
        pl.col("query_expand").alias("query_expand_long"),
    ])
    select_cols.extend(["query_expand_synonyms", "query_expand_long"])

gold_long = (
    gold.select(select_cols)
    .explode("docs")
    .with_columns([
        pl.col("docs").struct.field("publication_id").alias("publication_id"),
        pl.col("docs").struct.field("pmid").cast(pl.Utf8).alias("pmid"),
        pl.col("docs").struct.field("title").alias("title"),
        pl.col("docs").struct.field("abstract_clean").alias("abstract_clean"),
        pl.col("docs").struct.field("year").alias("year"),
        pl.col("docs").struct.field("anchor_pos").alias("anchor_pos"),
        pl.col("docs").struct.field("citation_captions").alias("citation_captions"),
    ])
    .drop("docs")
    .with_columns([
        pl.col("group_claim_id").cast(pl.Utf8),
        pl.col("pmid").cast(pl.Utf8),
    ])
 )

labeled = gold_long.join(labels, on=["group_claim_id", "pmid"], how="inner")

grouped = labeled.group_by("group_claim_id").agg([
    pl.first("query").alias("query"),
    pl.first("query_expand_synonyms").alias("query_expand_synonyms"),
    pl.first("query_expand_long").alias("query_expand_long"),
    pl.struct([
        "publication_id",
        "pmid",
        "title",
        "abstract_clean",
        "year",
        "anchor_pos",
        "citation_captions",
        "doc_match",
        "evidence_level",
        "reason",
    ]).alias("docs"),
])

PUBMED_URL_PREFIX = "http://www.ncbi.nlm.nih.gov/pubmed/"

questions = grouped.sort("group_claim_id").to_dicts()
for q in questions:
    pmids = [d.get("pmid") for d in q.get("docs", []) if d.get("pmid")]
    q["pmids"] = pmids
    # Canonical pipeline keys
    q["query_id"] = str(q.get("group_claim_id", ""))
    q["query_text"] = (q.get("query") or "").strip()
    q["body_expansion_synonyms"] = (q.get("query_expand_synonyms") or "").strip()
    q["body_expansion_long"] = (q.get("query_expand_long") or "").strip()
    q["documents"] = [PUBMED_URL_PREFIX + str(p) for p in pmids if p]

def _write_jsonl(path: Path, records) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

# Full payload (all fields) → private
_write_jsonl(OUT_JSONL_PRIVATE, questions)
print(f"Saved (private): {OUT_JSONL_PRIVATE}")

# Clean public: query_id, query_text, expansion variants, documents, docs
def to_public_question(q):
    return {
        "query_id": q["query_id"],
        "query_text": q["query_text"],
        "body_expansion_synonyms": q["body_expansion_synonyms"],
        "body_expansion_long": q["body_expansion_long"],
        "documents": q["documents"],
        "docs": q.get("docs", []),
    }

questions_public = [to_public_question(q) for q in questions]
_write_jsonl(OUT_JSONL, questions_public)
print(f"Saved (public): {OUT_JSONL}")

Saved (private): ../output/cleaned/dicty_gold_llm_private.json
Saved (public): ../output/cleaned/dicty_gold_llm_public.json


## Quick sanity checks

In [33]:
total_questions = len(questions)
total_docs = sum(len(q.get("docs", [])) for q in questions)

print(f"Questions: {total_questions}")
print(f"Labeled docs: {total_docs}")
print(f"Label source: {LABELS_PATH}")

Questions: 1656
Labeled docs: 2028
Label source: ../output/llama_full_agreement_cases.tsv


## Label stats

Percent breakdown for `doc_match` and `evidence_level` over labeled pairs.

In [34]:
def show_label_stats(df: pl.DataFrame, col: str) -> None:
    total = df.height
    if total == 0:
        print(f"{col}: no rows")
        return
    counts = (
        df.group_by(col)
        .len()
        .sort("len", descending=True)
        .with_columns((pl.col("len") / total * 100).round(2).alias("pct"))
    )
    print(f"\n{col} (n={total})")
    print(counts)

show_label_stats(labeled, "doc_match")
show_label_stats(labeled, "evidence_level")


doc_match (n=2028)
shape: (3, 3)
┌───────────┬──────┬───────┐
│ doc_match ┆ len  ┆ pct   │
│ ---       ┆ ---  ┆ ---   │
│ str       ┆ u32  ┆ f64   │
╞═══════════╪══════╪═══════╡
│ yes       ┆ 1860 ┆ 91.72 │
│ no        ┆ 160  ┆ 7.89  │
│ unclear   ┆ 8    ┆ 0.39  │
└───────────┴──────┴───────┘

evidence_level (n=2028)
shape: (3, 3)
┌──────────────────────────┬─────┬───────┐
│ evidence_level           ┆ len ┆ pct   │
│ ---                      ┆ --- ┆ ---   │
│ str                      ┆ u32 ┆ f64   │
╞══════════════════════════╪═════╪═══════╡
│ abstract_supports_detail ┆ 844 ┆ 41.62 │
│ abstract_supports_core   ┆ 762 ┆ 37.57 │
│ needs_fulltext           ┆ 422 ┆ 20.81 │
└──────────────────────────┴─────┴───────┘


## Export EPMC documents JSONL

Writes one JSON object per line from the cleaned EPMC abstracts parquet (key `abstract` = cleaned text).

In [20]:
# Export with key "abstract" (value = abstract_clean) so consumers (e.g. index scripts) use one field name.
docs = (
    pl.read_parquet(DOCS_PATH)
    .with_columns(pl.col("pmid").cast(pl.Utf8))
    .rename({"abstract_clean": "abstract"})
)
docs.write_ndjson(DOCS_JSONL_OUT)
print(f"Saved: {DOCS_JSONL_OUT}")

Saved: ../output/cleaned/articles_all_cleaned_abstract.jsonl
